# PRACTICE 3 — EXERCISE 1
**Nhóm 4 — Sentiment Analysis with Hugging Face**

Notebook chạy độc lập; thực hiện đủ 4 yêu cầu của Exercise 1. Lần đầu cần Internet để tải model. Chạy từ trên xuống, restart kernel nếu vừa cài thư viện. Kết quả được tạo khi chạy thực tế.

## 1.1 — Install the Hugging Face transformers library

In [1]:
%pip install "transformers==4.57.1" torch pandas

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.0 MB 381.6 kB/s eta 0:00:31
   -- ------------------------------------- 0.8/12.0 MB 606.8 kB/s eta 0:00:19
   ---- ----------------------------------- 1.3/12.0 MB 907.7 kB/s eta 0:00:12
   ---- ----------------------------------- 1.3/12.0 MB 907.7 kB/s eta 0:00:12
   ---- ----------------------------------- 1.3/12.0 MB 907.7 kB/s eta 0:00:12
   ------ --------------------------------- 1.8/12.0 MB 883.3 kB/s eta 0:00:12
   ------- -------------------------------- 2.4/12.0 MB 1.1 MB/s eta 0:00:10
   ------- -------------------------------- 2.4/12.0 MB 1.1 MB/s eta 0:00:10
   ------- -----------

In [2]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import gc
from datetime import datetime
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display
from transformers import set_seed
set_seed(42)
CONFIG = {"exercise1_model": "distilbert/distilbert-base-uncased-finetuned-sst-2-english"}
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "configs" / "default.json").is_file() and (p / "src").is_dir()), Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / "outputs" / f"exercise1_{datetime.now():%Y%m%d_%H%M%S_%f}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU CUDA:", torch.cuda.is_available())
print("Kết quả lưu ở:", OUTPUT_DIR)

GPU CUDA: False
Kết quả lưu ở: c:\Users\Hoang Thien (Sora)\Downloads\Lab3_DL_Group4-main\outputs\exercise1_20261119_003325_529427


## 1.2 — Use a pretrained sentiment analysis model from the Hub

### Tải pretrained model và tokenizer

Dùng checkpoint `distilbert/distilbert-base-uncased-finetuned-sst-2-english`. Nếu dùng lần đầu cần Internet. Model được chuyển sang GPU nếu có, ngược lại dùng CPU.

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

EX1_MODEL = CONFIG["exercise1_model"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    ex1_tokenizer = AutoTokenizer.from_pretrained(EX1_MODEL)
    ex1_model = AutoModelForSequenceClassification.from_pretrained(EX1_MODEL)
except Exception as exc:
    raise RuntimeError(
        f"Không tải được pretrained model Exercise 1: {EX1_MODEL}. "
        "Kiểm tra Internet / Hugging Face Hub."
    ) from exc
ex1_model.to(device).eval()
print("Model:", EX1_MODEL)
print("Thiết bị:", device)
print("id2label:", ex1_model.config.id2label)

Model: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Thiết bị: cpu
id2label: {0: 'NEGATIVE', 1: 'POSITIVE'}


## 1.3 — Tokenize a sample sentence

Quan sát `input_ids`, `attention_mask` và chuỗi token để giải thích vai trò tokenizer và token đặc biệt `[CLS]`, `[SEP]`.

In [4]:
SAMPLE_TEXT = "I love this movie. The acting is amazing!"
tokenized_sample = ex1_tokenizer(SAMPLE_TEXT, return_tensors="pt", truncation=True)
print("Câu mẫu:", SAMPLE_TEXT)
print("Tokens:", ex1_tokenizer.convert_ids_to_tokens(tokenized_sample["input_ids"][0]))
print("input_ids:", tokenized_sample["input_ids"][0].tolist())
print("attention_mask:", tokenized_sample["attention_mask"][0].tolist())
print("Shape:", tuple(tokenized_sample["input_ids"].shape))

Câu mẫu: I love this movie. The acting is amazing!
Tokens: ['[CLS]', 'i', 'love', 'this', 'movie', '.', 'the', 'acting', 'is', 'amazing', '!', '[SEP]']
input_ids: [101, 1045, 2293, 2023, 3185, 1012, 1996, 3772, 2003, 6429, 999, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Shape: (1, 12)


## 1.4 — Perform sentiment analysis on the sentence

`softmax(logits)` chuyển logits thành phân phối xác suất; `argmax` xác định lớp dự đoán. Score là xác suất đầu ra của mô hình, không nhất thiết là độ tin cậy đã được hiệu chuẩn.

Đầu tiên suy luận trên chính `tokenized_sample` ở bước 1.3, sau đó thử thêm các câu đối lập.

In [5]:
# Sử dụng chính dữ liệu tokenized từ bước 1.3.
sample_inputs = {key: value.to(device) for key, value in tokenized_sample.items()}
with torch.inference_mode():
    sample_probs = torch.softmax(ex1_model(**sample_inputs).logits, dim=-1)[0].cpu()
sample_idx = int(sample_probs.argmax())
print("Câu mẫu:", SAMPLE_TEXT)
print("Sentiment:", ex1_model.config.id2label[sample_idx])
print("Score:", round(float(sample_probs[sample_idx]), 6))
print("Xác suất từng nhãn:", {ex1_model.config.id2label[i]: round(float(v), 6)
                              for i, v in enumerate(sample_probs)})

Câu mẫu: I love this movie. The acting is amazing!
Sentiment: POSITIVE
Score: 0.999881
Xác suất từng nhãn: {'NEGATIVE': 0.000119, 'POSITIVE': 0.999881}


In [6]:
example_texts = [
    SAMPLE_TEXT,
    "This movie is extremely boring and disappointing.",
    "The visuals were good, although the story was weak.",
]
ex1_inputs = ex1_tokenizer(example_texts, padding=True, truncation=True,
                           max_length=256, return_tensors="pt")
ex1_inputs = {name: tensor.to(device) for name, tensor in ex1_inputs.items()}
with torch.no_grad():
    probs = torch.softmax(ex1_model(**ex1_inputs).logits, dim=-1).cpu().numpy()
pred_ids = probs.argmax(axis=1)
ex1_results = pd.DataFrame({
    "text": example_texts,
    "predicted_label": [ex1_model.config.id2label[int(i)] for i in pred_ids],
    "score": [round(float(probs[k, i]), 6) for k, i in enumerate(pred_ids)],
})
display(ex1_results)
ex1_results.to_csv(OUTPUT_DIR / "exercise1_predictions.csv", index=False, encoding="utf-8-sig")

# Giải phóng mô hình Exercise 1 trước khi tạo model train của Exercise 2.
del ex1_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

,text,predicted_label,score
0,I love this movie. The acting is amazing!,POSITIVE,0.999881
1,This movie is extremely boring and disappointing.,NEGATIVE,0.999794
2,"The visuals were good, although the story was ...",POSITIVE,0.999764


## Kết luận
Mô hình DistilBERT đã fine-tune trên SST-2 được dùng để tokenize và dự đoán cảm xúc, không huấn luyện thêm. Score là xác suất đầu ra của mô hình, không phải accuracy trên tập kiểm thử.

[Tham khảo Hugging Face](https://huggingface.co/blog/sentiment-analysis-python).